<a href="https://colab.research.google.com/github/Ankita456640/ml_lab/blob/main/CarPricePredictionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import sklearn.metrics as metrics
from sklearn.metrics import r2_score

In [ ]:
import pandas as pd
from google.colab import files
import os


file_name = 'car_price_predisction.csv'

if not os.path.exists(file_name):
    print(f"File '{file_name}' not found. Please upload it.")
    uploaded = files.upload()
    if file_name not in uploaded:
        print(f"'{file_name}' was not uploaded. Please ensure you upload the correct file.")

df = pd.read_csv(file_name)
print(df.head())

File 'car_price_predisction.csv' not found. Please upload it.


In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
#merge brand and model
df['brand_model'] = df['Brand'] + ' ' + df['Model']
print(df['brand_model'])

In [ ]:
df.drop(['Model', 'Brand'], axis=1, inplace=True)
print(df.head())

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

if 'brand_model' not in df.columns:
    df['brand_model'] = df['Brand'] + ' ' + df['Model']

df['brand_model_encoded'] = le.fit_transform(df['brand_model'])

In [ ]:
print(df.head())
print(df.tail())

In [ ]:
#graph plotting year against milage

plt.hist(df['Price'],bins=100, color='LightPink')
plt.xlabel('Year')
plt.ylabel('Price')
plt.title('Year vs Price')
plt.show()

In [ ]:
#car price median acording to engine and milage
df['median_price'] = df.groupby(['Engine Size', 'Mileage', 'Year'])['Price'].transform('median')
print(df[['Engine Size', 'Mileage', 'Year', 'Price', 'median_price']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.histplot(df['median_price'], kde=True, stat="count", color="pink", ax=axes[0])
axes[0].set_xlabel("Median Price")
axes[0].set_title("Method 1: Seaborn (Automatic Fit)")

count, bins, ignored = axes[1].hist(df['median_price'], bins=30, density=True,
                                    alpha=0.5, color='purple', edgecolor='black',
                                    label='Count Histogram')
from scipy.stats import norm

mu, std = norm.fit(df['median_price'])
x = np.linspace(min(df['median_price']), max(df['median_price']), 100)
p = norm.pdf(x, mu, std)
axes[1].plot(x, p, 'r', linewidth=2, label='Fitted Normal Curve')

axes[1].set_title("Method 2: Matplotlib (Manual Fit)")
axes[1].set_ylabel("median_price")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#heatmap for correlation between brand_model and price

corr_matrix = df[['brand_model_encoded', 'Price']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='Pastel1')
plt.title("Correlation Heatmap: Encoded Brand Model vs Price")
plt.show()

In [ ]:
df.info()

In [ ]:
#handling or finding missing data
df.isnull().sum()

In [ ]:
#handling duplicae datas
df.duplicated().sum()

In [ ]:
df.to_csv('cleaned_car_price_prediction.csv', index=False)

In [ ]:
#handling inconsistency
columns_to_check = ['Fuel Type', 'Engine Size', 'Transmission', 'Mileage', 'median_price']
for col in columns_to_check:
    print(f"Unique values for '{col}':")
    print(df[col].unique())

In [ ]:
#handling garbage data

for i in df.select_dtypes(include=['object']).columns:
    print(df[i].value_counts())


In [ ]:
#standardisation
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaler.fit(df[['Engine Size', 'Mileage', 'Year', 'brand_model_encoded']])

In [ ]:
standard_scaler=pd.DataFrame(scaler.transform(df[['Engine Size', 'Mileage', 'Year', 'brand_model_encoded']]),
                             columns=(['Engine Size', 'Mileage', 'Year', 'brand_model_encoded']),
                             index=df.index)

In [ ]:
standard_scaler.head()

In [ ]:
standard_scaler.describe()

In [ ]:
standard_scaler.std()

In [ ]:
standard_scaler.hist(figsize=(10,10), bins=30, color='pink')
plt.show()

# **EDA**

exploratory Data Analysis



In [ ]:
#describing data
df.describe().T

In [ ]:
df.drop(['Car ID'], axis=1, inplace=True)

In [ ]:
df.hist(figsize=(10,10), bins=30, color='pink')
plt.show()

## **data separation in two parts**





In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

In [ ]:
list_indices = list(split.split(df, df['brand_model_encoded']))

In [ ]:
list_indices

In [ ]:
train_index, test_index = list_indices[0]

In [ ]:
X_train = df.iloc[train_index]
X_test = df.iloc[test_index]

In [ ]:
len(X_train), len(X_test)

In [ ]:
x_train=X_train['brand_model_encoded'].value_counts().sort_index() / len(X_train)

In [ ]:
x_train

In [ ]:
x_test=X_test['brand_model_encoded'].value_counts().sort_index() / len(X_test)

In [ ]:
x_test

In [ ]:
x_train=X_train.drop('median_price', axis=1)
y_train=X_train['median_price'].copy()

In [ ]:
x_test=X_test.drop('median_price', axis=1)
y_test=X_test['median_price'].copy()

In [ ]:
X_train = X_train.drop('median_price', axis=1)
X_test = X_test.drop('median_price', axis=1)

In [ ]:
X_train.info()

In [ ]:
x_train

In [ ]:
plt.scatter(X_train['brand_model_encoded'], y_train, color='pink')
plt.xlabel('Brand Model Encoded')
plt.ylabel('Median Price')
plt.title('Brand Model Encoded vs Median Price')
plt.show()

### **One-Hot encoding**

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
X_train_ohe = ohe.fit_transform(X_train)

In [ ]:
X_train_ohe.toarray()[:5]

In [ ]:
type(X_train_ohe)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numerical_features = ['Year', 'Engine Size', 'Mileage', 'Price', 'brand_model_encoded']
categorical_features = ['Fuel Type', 'Transmission', 'Condition']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='drop'
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

In [ ]:
X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)

The `X_train_processed` and `X_test_processed` are now NumPy arrays containing the scaled numerical features and one-hot encoded categorical features. These are ready for use in machine learning models.

In [ ]:
print(X_train_processed[:5])

# **Model prediction**

### ***Linear regression***

In [ ]:
#multiple linear regression
from sklearn.linear_model import LinearRegression

linear_reg_model = LinearRegression()
linear_reg_model.fit(X_train_processed, y_train)

In [ ]:
linear_reg_model.predict(X_train_processed[0:1])

In [ ]:
y_train[0]

In [ ]:
y_pred= linear_reg_model.predict(X_train_processed)
y_pred

In [ ]:
import sklearn.metrics as metrics
from sklearn.metrics import mean_squared_error
y_pred = linear_reg_model.predict(X_train_processed)
mse = mean_squared_error(y_train, y_pred)
rmse = np.sqrt(mse)

In [ ]:
mse

In [ ]:
rmse, print(rmse)

In [ ]:
prediction= pd.DataFrame({'brand_model': X_train['brand_model'],
                          'Milage': X_train['Mileage'],
                          'Year': X_train['Year'],
                          'Predicted_Price': y_pred})
prediction

###**k-fold cross validation**

K-fold is good for model selection and hyperparameter tuning. it also helps to avoid overfitting.

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(linear_reg_model, X_train_processed, y_train, scoring='neg_mean_squared_error', cv=10)

In [ ]:
scores

**Evaluation of model**

In [ ]:
import sklearn.metrics as metrics
from sklearn.metrics import mean_squared_error

y_pred_test = linear_reg_model.predict(X_test_processed)

mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)

In [ ]:
mse

In [ ]:
rmse, print(rmse)

In [ ]:
#compare baseline
from sklearn.dummy import DummyRegressor
dummy_regressor = DummyRegressor(strategy='mean')
dummy_regressor.fit(X_train, y_train)
y_pred_dummy = dummy_regressor.predict(X_test)
mse_dummy = mean_squared_error(y_test, y_pred_dummy)
rmse_dummy = np.sqrt(mse_dummy)

In [ ]:
mse_dummy

In [ ]:
mse_model = mse
rmse_model = rmse

In [ ]:
rmse_model

In [ ]:
#residual plot
residual= y_test - y_pred_test
plt.scatter(y_pred_test, residual, color='pink')
plt.axhline(y=0, color='b', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_train, linear_reg_model.predict(X_train_processed))
rmse = np.sqrt(mse)
rmse

In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_train, y_pred)
rmse = np.sqrt(mse)
rmse

In [ ]:
from sklearn.linear_model import Ridge
model_ridge = Ridge(alpha=0.5) #Ridge regularisation helps to solve overfitting issue too.
model_ridge.fit(X_train_processed, y_train)

In [ ]:
#boxplot of features and price
plt.figure(figsize=(10, 6))
sns.boxplot(x='brand_model_encoded', y='Price', data=df, color='pink')
plt.xlabel('Brand Model Encoded')
plt.ylabel('Price')
plt.title('Boxplot of Brand Model Encoded vs Price')
plt.show()

In [ ]:
#iqr method
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_filtered = df[(df['Price'] >= lower_bound) & (df['Price'] <= upper_bound)]
df_filtered.sum()

In [ ]:
plt.boxplot(df['Price'])
plt.ylabel('Price')
plt.title('Boxplot of Price')
plt.show()

In [ ]:
model_ridge = Ridge(alpha=0.0001)
model_ridge.fit(X_train_processed, y_train)

In [ ]:
Train_rmse = np.sqrt(mean_squared_error(y_train, linear_reg_model.predict(X_train_processed)))
Test_rmse = np.sqrt(mean_squared_error(y_test, model_ridge.predict(X_test_processed)))

In [ ]:
Train_rmse

In [ ]:
Test_rmse

In [ ]:
'''from sklearn.preprocessing import PolynomialFeatures
poly_features = PolynomialFeatures(degree=2)
X_train_poly = poly_features.fit_transform(X_train_processed)
linear_reg_model_poly = LinearRegression()
linear_reg_model_poly.fit(X_train_poly, y_train)'''    #polynomial feature is almost useless here as the model is linear in nature. also it can cause overfitting.

In [ ]:
'''residual= y_test - linear_reg_model_poly.predict(poly_features.transform(X_test_processed))
plt.scatter(y_pred_test, residual, color='pink')
plt.axhline(y=0, color='b', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()'''

In [ ]:
##poly_train_rmse = np.sqrt(mean_squared_error(y_train, linear_reg_model_poly.predict(X_train_poly)))

In [ ]:
#poly_test_rmse = np.sqrt(mean_squared_error(y_test, linear_reg_model_poly.predict(poly_features.transform(X_test_processed))))

In [ ]:
#poly_train_rmse

In [ ]:
#poly_test_rmse

In [ ]:
#pickling
import pickle
with open('model.pkl','wb') as f:
  pickle.dump(linear_reg_model,f)

In [ ]:
import pickle
with open('model.pkl','rb') as f:
  model=pickle.load(f)